# Playing Card Detection - YOLOv8 Training

This notebook trains a YOLOv8 model for detecting playing cards (52 classes: all ranks and suits).

**Dataset:** Roboflow Playing Cards
- Training: 21,203 images
- Validation: 2,020 images
- Classes: 52 (10C, 10D, 10H, 10S, ..., QC, QD, QH, QS)

## 1. Setup and Imports

In [ ]:
# Install required packages (if needed)
# !pip install ultralytics

import os
from pathlib import Path
import yaml
import shutil
from ultralytics import YOLO
import torch
from IPython.display import Image, display

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Dataset Configuration

In [ ]:
# Project paths - updated for backend directory location
PROJECT_ROOT = Path.cwd().parent  # Go up one level from backend/
BACKEND_DIR = Path.cwd()  # backend directory
DATASET_DIR = PROJECT_ROOT / 'datasets'
DATA_YAML = DATASET_DIR / 'data.yaml'
MODELS_DIR = BACKEND_DIR / 'models'  # Store trained models here

print(f"Project root: {PROJECT_ROOT}")
print(f"Backend directory: {BACKEND_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Data config: {DATA_YAML}")

# Verify dataset structure
assert DATASET_DIR.exists(), f"Dataset directory not found: {DATASET_DIR}"
assert DATA_YAML.exists(), f"data.yaml not found: {DATA_YAML}"

train_images = DATASET_DIR / 'train' / 'images'
valid_images = DATASET_DIR / 'valid' / 'images'

train_count = len(list(train_images.glob('*')))
valid_count = len(list(valid_images.glob('*')))

print(f"\nDataset verified:")
print(f"  Training images: {train_count:,}")
print(f"  Validation images: {valid_count:,}")

In [ ]:
# Training hyperparameters
MODEL_SIZE = 'yolov8n.pt'  # nano (fastest), also try: yolov8s.pt, yolov8m.pt
EPOCHS = 100
BATCH_SIZE = 16  # Adjust based on GPU memory (8, 16, 32, 64)
IMG_SIZE = 640
PATIENCE = 20  # Early stopping patience

# Output directory - save in backend/runs
RUNS_DIR = BACKEND_DIR / 'runs'
RUNS_DIR.mkdir(exist_ok=True)

print("Training Configuration:")
print(f"  Model: {MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Patience: {PATIENCE}")
print(f"  Output: {RUNS_DIR}")

## 5. Load Pretrained Model

In [ ]:
# Load pretrained YOLOv8 model
model = YOLO(MODEL_SIZE)
print(f"Loaded pretrained model: {MODEL_SIZE}")
print(f"Model summary:")
model.info()

## 6. Train Model

This will train the model on the playing cards dataset. Training may take 2-6 hours depending on your GPU.

## 3. Training Configuration & Model Training

In [ ]:
# Read current data.yaml and update paths to absolute
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print("Original data.yaml:")
print(f"  train: {data_config['train']}")
print(f"  val: {data_config['val']}")
print(f"  test: {data_config.get('test', 'N/A')}")
print(f"  nc: {data_config['nc']}")
print(f"  names: {len(data_config['names'])} classes")

# Update to absolute paths
data_config['train'] = str(DATASET_DIR / 'train' / 'images')
data_config['val'] = str(DATASET_DIR / 'valid' / 'images')
data_config['test'] = str(DATASET_DIR / 'test' / 'images')

# Save updated config in backend directory
data_yaml_fixed = BACKEND_DIR / 'data_fixed.yaml'
with open(data_yaml_fixed, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"\nUpdated config saved to: {data_yaml_fixed}")
print(f"  train: {data_config['train']}")
print(f"  val: {data_config['val']}")

# Train the model
results = model.train(
    data=str(data_yaml_fixed),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    save=True,
    device=0 if torch.cuda.is_available() else 'cpu',  # Use GPU if available
    workers=8,  # Data loading workers
    project=str(RUNS_DIR),
    name='card_detection',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    deterministic=True,
    val=True,
    plots=True,
)

print("\nTraining completed!")

In [ ]:
# Validate the trained model
metrics = model.val()

print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 8. Test Predictions

Test the trained model on sample images

In [ ]:
# Load the best trained model
best_model_path = RUNS_DIR / 'card_detection' / 'weights' / 'best.pt'
trained_model = YOLO(str(best_model_path))

print(f"Loaded best model: {best_model_path}")

# Test on validation images
test_images = list(valid_images.glob('*.jpg'))[:5]  # First 5 images

for img_path in test_images:
    results = trained_model(str(img_path))
    
    # Display results
    for result in results:
        print(f"\nImage: {img_path.name}")
        print(f"  Detected {len(result.boxes)} cards")
        
        # Show detections
        for box in result.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            label = result.names[cls]
            print(f"    {label}: {conf:.2%}")
        
        # Save annotated image
        result.save(filename=f"test_result_{img_path.name}")

print("\nTest predictions saved!")

## 9. Export Model for Production

## 10. Model Analysis & Metrics Visualization

In [ ]:
# Display training metrics
results_dir = RUNS_DIR / 'card_detection'

print("Training Results:")
print(f"  Location: {results_dir}")

# Display key plots
plots = [
    'results.png',      # Training metrics over time
    'confusion_matrix.png',  # Confusion matrix
    'val_batch0_pred.jpg',   # Validation predictions
]

for plot in plots:
    plot_path = results_dir / plot
    if plot_path.exists():
        print(f"\n{plot}:")
        display(Image(filename=str(plot_path)))
    else:
        print(f"\n{plot} not found")

In [ ]:
# Copy best model to backend/models directory
best_model_path = RUNS_DIR / 'card_detection' / 'weights' / 'best.pt'
final_model_path = MODELS_DIR / 'best.pt'

if best_model_path.exists():
    shutil.copy(best_model_path, final_model_path)
    print(f"Model copied to: {final_model_path}")
    print("\nTo use this model in the backend:")
    print("  1. Set environment variable: MODEL_PATH=models/best.pt")
    print("  2. Or update services/model_service.py to use 'models/best.pt'")
    print("  3. Restart the backend server")
    print("  4. The backend will automatically load the trained model")
    print(f"\nModel is ready at: {final_model_path.relative_to(BACKEND_DIR)}")
else:
    print(f"Best model not found at: {best_model_path}")
    print("Make sure training completed successfully")

## Summary

**Training completed!**

Next steps:
1. Review training metrics above
2. Test the model on sample images
3. Copy `best.pt` to backend directory
4. Update backend MODEL_PATH environment variable
5. Test real-time detection with webcam

**Model Location:** `runs/card_detection/weights/best.pt`